# MIMORI: experimental Laya decision-head fine-tuning

Runs the repository CPU trainer with a frozen encoder. Synthetic source families are separated before augmentation. This is supervised training, not RL, GPU/DDP or an independent benchmark. Enable Internet in your notebook host.

In [ ]:
from pathlib import Path
import os, subprocess
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
repo = work / 'MIMORI'
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/j33v4nz/MIMORI.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

## Install

Use a dedicated runtime. Restart its kernel if requested after installation.

In [ ]:
%pip install 'torch>=2.6,<3' --index-url https://download.pytorch.org/whl/cpu
%pip install -r scripts/laya/requirements.txt

## Generate and validate

300 samples demonstrate the pipeline. Add representative labeled workloads for further evaluation. Preserve exact datasets and hashes.

In [ ]:
subprocess.run(['python', 'scripts/laya/dataset_generator.py', '--count', '300', '--seed', '42', '--output-dir', 'scripts/laya/data'], check=True)
subprocess.run(['python', 'scripts/laya/train.py', '--dry-run'], check=True)

## Train real weights

The pinned base requires roughly 842 MB. CPU execution can take over an hour. Successful training exports a manifest with actual optimizer steps and losses.

In [ ]:
subprocess.run(['python', 'scripts/laya/train.py', '--epochs', '6', '--batch-size', '8', '--lr', '0.0001', '--threads', '4'], check=True)

## Evaluate held-out rows

Evaluation requires a loaded model and rejects heuristic fallback. Inspect class supports, false positives, calibration and hardware latency.

In [ ]:
subprocess.run(['python', 'scripts/laya/evaluate.py', '--data-file', 'scripts/laya/data/test.jsonl', '--model-path', 'scripts/laya/checkpoint', '--output', 'scripts/laya/checkpoint/evaluation.json', '--threads', '4'], check=True)
print(Path('scripts/laya/checkpoint/evaluation.json').read_text())

## Export exact artifacts

Include upstream attribution/license and a model card before redistributing. Use `serve.py --require-model` and verify health reports `loaded_model: true`.

In [ ]:
import tarfile, hashlib
archive = work / 'mimori-laya-experimental.tar.gz'
with tarfile.open(archive, 'w:gz') as tar:
    tar.add('scripts/laya/checkpoint', arcname='checkpoint')
    tar.add('scripts/laya/data', arcname='data')
print(archive)
print('SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())